# Week 3: Edge Detection

This python notebook includes all the exercises from this session. All the needed setup, I/O and user-interaction code is already provided to let you concentrate on this week's topic.<br>
Please run all the provided code in order before running yours.<br>

This week's assignment:

## 1. Thresholded Edge Detection

Implement a function that takes as input the derivative kernels and threshold and outputs the edge image of the provided test image (lighthouse.png). Then test its output with different kernels and thresholds to compare the results. As a reference we report some suggested kernels:

Elementary derivative kernels:
$$ k_x = \left[ {\begin{array}{cc}
   -1 & 1 \\
  \end{array} } \right]
  ~~~
   k_y = \left[ {\begin{array}{c}
   -1\\
   1\\
  \end{array} } \right]$$

Sobel derivative kernels:
$$ k_x = \left[ {\begin{array}{ccc}
   -1 & 0 & 1 \\
   -2 & 0 & 2 \\
   -1 & 0 & 1 \\
  \end{array} } \right]
  ~~~
   k_y = \left[ {\begin{array}{ccc}
   -1 & -2 & -1 \\
   0 & 0 & 0 \\
   1 & 2 & 1 \\
  \end{array} } \right]$$

Prewitt derivative kernels:
$$ k_x = \left[ {\begin{array}{ccc}
   -1 & 0 & 1 \\
   -1 & 0 & 1 \\
   -1 & 0 & 1 \\
  \end{array} } \right]
  ~~~
   k_y = \left[ {\begin{array}{ccc}
   -1 & -1 & -1 \\
   0 & 0 & 0 \\
   1 & 1 & 1 \\
  \end{array} } \right]$$

Inside the function, compute the gradient images in $x$ and $y$ direction and consider a pixel $(x,y)$ to be part of an edge if $\sqrt{g_x(x,y)^2+g_y(x,y)^2}>threshold$ holds.

## 2. Canny Edge Detection

Implement edge detection using the Canny method.

1. Produce and show kernels composing derivative and gaussian blur for noise reduction
2. Produce the gradient images and show the gradient magnitude image, where each pixel contains the norm of the gradient
3. Produce the quantized gradient direction image
    1. Produce the gradient direction image, where each pixel contains the angle of the gradient
    2. Produce and show the quantized gradient image, where each pixel is an integer corresponding to a gradient direction (horizontal, primary diagonal, vertical, secondary diagonal)
4. Perform nonmaximum suppression on the gradient magnitude image for edge thinning, and show it. For each pixel, it is taken only if it is the maximum with the nearby pixels along its corresponding gradient direction.
5. Perform hysteresis on the resulting image, and show the final result of Canny Edge Detection.
    1. Select two thresholds $t_{high}$ and $t_{low}$
    2. A pixel on the nonmaximum suppressed gradient image is taken if its value is higher than $t_{low}$ and there exists any path through pixels with value higher than $t_{low}$ that connects it to a pixel with value higher than $t_{high}$.
6. Identify all the hyperparameters introduced in this exercise, play with them and understand their role. To facilitate this, in this tutorial we provide a fill-in template to change your hyperparameters with interactive sliders and visualize the output.

## 3. RANSAC Line Fitting

An edge map is full of outliers: a least-squares line fitted to all edge pixels is pulled away by every pixel that does not belong to the line. RANSAC (RANdom SAmple Consensus) estimates a model robustly by repeating the following steps:

1. Randomly sample a minimal set of points (2 points for a line) and fit a candidate line to them.
2. Count the inliers of the candidate, i.e. all points whose distance to the line is below a threshold $\tau$.
3. Keep the candidate with the largest number of inliers.

Finally, the line is re-estimated with least squares on the inliers of the best candidate only.

We represent a line in normal form $\mathbf{n}^\top \mathbf{p} + d = 0$ with $\|\mathbf{n}\| = 1$, where $\mathbf{p} = (x, y)^\top$. The distance of a point $\mathbf{p}$ to the line is then simply $|\mathbf{n}^\top \mathbf{p} + d|$. Unlike $y = mx + b$, this representation can also describe vertical lines.

1. Fill in the functions that fit a line through two points, compute point-to-line distances and fit a line to many points with (total) least squares.
2. Fill in the RANSAC loop.
3. Compute how many iterations $N$ are needed to draw at least one outlier-free sample with probability $p$, given an outlier ratio $e$ and a sample size $s$: $N = \frac{\log(1-p)}{\log\left(1-(1-e)^s\right)}$
4. Compare RANSAC with plain least squares on synthetic points with outliers.
5. Detect the dominant straight lines in the edge image of lighthouse.png by running RANSAC repeatedly, removing the inliers of each detected line. Look at the result: which lines are real, which are not, and why?

## Useful information for this tutorial:

For basics about images as numpy arrays, please refer to the second week's tutorial.

With numpy it is also possible to refer to slices of a given array by specifying the kept dimensions in the slice with `:`. For example, if we want to access the red channel of an RGB image `im` with shape `(height, width, 3)`, we can write `im[:, :, 0]`, which refers to the whole image data for the red channel only. We can optionally specify a range to be kept along a direction by writing the extreme indices around the `:` as in `min:max`. For example, accessing a patch within the pixel coordinates `(h0, w0)`, `(h1, w1)` with h0 < h1 and w0 < w1 can be done with `im[h0:h1, w0:w1, :]`.<br>
You can also index arrays based on conditions: for example to set to black all pixels with a blue component higher than 200 you can write `im[im[:, :, 2] > 200] = 0`.<br>
These references can be used both for reading an image slice and for writing a whole slice at once, without looping over it. <br>
A complete reference of numpy basic and advanced indexing rules (not necessary for this tutorial, but very useful for numpy python programming) can be found in https://numpy.org/doc/stable/user/basics.indexing.html

When doing operations between numpy arrays, all operations are executed element-wise (except for the `@` operator that performs the usual matrix multiplication). This requires the shapes (dimensions) of the involved arrays to match or to be compatible according to broadcasting rules, which are usually intuitive. An example of broadcasting is when adding a pixel value `p=np.array([70, 128, 255])` to an image `im` with shape `(height, width, 3)`: the operation is legal and the pixel value is added independently to each pixel in the image. <br>
A complete reference of numpy broadcasting rules (not necessary for this tutorial, but very useful for numpy python programming) can be found in https://numpy.org/doc/stable/user/basics.broadcasting.html .

In the following we list some useful functions.

Array initialization:<br>
`np.array(a, dtype)`: returns a copy of `a` as a numpy array. Argument `a` does not need to be a numpy array, it can be any array-like object, e.g. a python list. It is possible to specify a `dtype` argument in case we want to change the type of the original array in the returned copy. https://numpy.org/doc/stable/reference/generated/numpy.array.html<br>
`np.empty(shape, dtype)`: returns a new empty array with specified shape `shape` and the specified data type `dtype`. For this tutorial we suggest using as dtype `np.int16` for images and default for the rest. https://numpy.org/doc/stable/reference/generated/numpy.empty.html<br>
`np.ones(shape, dtype)`: returns a new array filled with ones with specified shape `shape` and the specified data type `dtype`. For this tutorial we suggest using as dtype `np.int16` for images and default for the rest. https://numpy.org/doc/stable/reference/generated/numpy.ones.html<br>
`np.zeros(shape, dtype)`: returns a new array filled with zeros with specified shape `shape` and the specified data type `dtype`. For this tutorial we suggest using as dtype `np.int16` for images and default for the rest. https://numpy.org/doc/stable/reference/generated/numpy.zeros.html<br>
`np.eye(N)` : builds the identity matrix of order `N`. https://numpy.org/doc/stable/reference/generated/numpy.eye.html<br>

Array shape management:<br>
`np.reshape(a, newshape)`: returns a reshaped view of the input array `a` having shape `newshape`, which should be specified as a tuple of integers. https://numpy.org/doc/stable/reference/generated/numpy.reshape.html<br>
`np.transpose(a)`: returns the transposed of array `a`, whose shape has reversed order with respect to `a`. Alias `a.T`. https://numpy.org/doc/stable/reference/generated/numpy.transpose.html<br>
`np.expand_dims(a, axis)`: returns the same array `a` with a new dimension of size one along the specified `axis`. You can use `np.newaxis` when slice-indexing `a` for the same result. https://numpy.org/doc/stable/reference/generated/numpy.expand_dims.html<br>

Array reductors:<br>
`np.linalg.norm(x)`: returns the L2 norm of array `x`. You can also specify a keyword argument `axis` to compute the norm only along a specific dimension (the result will have the same shape as `x`, but without the specified dimension) https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html<br>
`np.sum(a)` : returns the sum of all the elements of array `a`. You can specify a keyword argument `axis` to compute the sum only along a specific dimension (the result will have the same shape as `a`, but without the specified dimension) https://numpy.org/doc/stable/reference/generated/numpy.sum.html<br>
`np.mean(a)` : returns the mean value of all the elements of array `a`. You can specify a keyword argument `axis` to compute the mean only along a specific dimension (the result will have the same shape as `a`, but without the specified dimension) https://numpy.org/doc/stable/reference/generated/numpy.mean.html<br>

Matrix operations:<br>
`np.linalg.inv(a)` : returns the multiplicative inverse of matrix `a`. https://numpy.org/doc/stable/reference/generated/numpy.linalg.inv.html<br>
`np.matmul(a, b)` : returns the matrix multiplication of `a` multiplied by `b`. Alias `a @ b`. https://numpy.org/doc/stable/reference/generated/numpy.matmul.html<br>

Image operations:<br>
`scipy.ndimage.convolve(input, weights)`: returns the convolution of `input` image using the `weights` image as kernel. https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.convolve.html<br>
`scipy.ndimage.shift(input, shift)`: returns a copy of `input` image shifted by the translation vector `shift`. Argument `shift` is in pixels, it can be not integer, in which case the result is interpolated. https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.shift.html<br>
`scipy.ndimage.label(input, structure)`: Segment the connected components in `input` image, considering all zero-values as separation elements. The `structure` parameter is a central-symmetric array with ones and zeros that defines when two parts are to be considered connected or not (i.e. component_1 is connected with component_2 if we can center the `structure` array on a pixel in component_1 and have at least a one in `structure` to overlap with a pixel in component_2). This function returns both the segmented image, assigning to each pixel an integer identifying its own connected component label, and the number of distinct connected components found. https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.label.html<br>

Boolean array operations:<br>
`np.logical_and(x1, x2)`: where `x1` and `x2` are arrays of boolean values with matching or broadcastable shape, it returns the element-wise logical and operation between `x1` and `x2`. https://numpy.org/doc/stable/reference/generated/numpy.logical_and.html<br>
`np.isin(element, test_elements)`: for each element in array `element` check whether it is contained into the array-like `test_elements`, and returns an array of boolean values of the same shape as `element`. Warning: with non-sequence `test_elements` like python sets, it produces incorrect results without throwing any exception. https://numpy.org/doc/stable/reference/generated/numpy.isin.html<br>

Miscellaneous:<br>
`np.around(a, decimals)`: rounds a floating-point array `a` to the specified number of `decimals` (zero decimals by default). Returns again a floating point array. https://numpy.org/doc/stable/reference/generated/numpy.around.html<br>
`np.maximum(x1, x2)`: where `x1` and `x2` are arrays with matching or broadcastable shape, it returns the element-wise maximum into an array with the same (or broadcasted) shape as the inputs. https://numpy.org/doc/stable/reference/generated/numpy.maximum.html<br>
`np.pi`: the usual constant for the ratio between circumference and diameter of a circle. https://numpy.org/doc/stable/reference/constants.html#numpy.pi<br>
`np.arctan2(x1, x2)`: element-wise arctan of `x1/x2` choosing the quadrant correctly. Values returned are in the range `[-pi, pi]`. https://numpy.org/doc/stable/reference/generated/numpy.arctan2.html<br>
`np.hypot(x1, x2)`: element-wise `sqrt(x1**2 + x2**2)` for calculating the gradient magnitude. https://numpy.org/doc/stable/reference/generated/numpy.hypot.html<br>

Random numbers and linear algebra (Exercise 3):<br>
`np.random.default_rng(seed)`: returns a random generator `rng`. Fixing the `seed` makes your results reproducible. https://numpy.org/doc/stable/reference/random/generator.html<br>
`rng.choice(a, size, replace=False)`: draws `size` distinct random elements from `np.arange(a)` (if `a` is an integer) or from the array `a`. https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html<br>
`np.linalg.svd(a)`: returns the singular value decomposition `U, S, Vt` of matrix `a`. The rows of `Vt` are sorted by decreasing singular value, so `Vt[-1]` is the direction in which the data varies the least. https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html<br>
`np.nonzero(a)`: returns the indices of the non-zero elements of `a`, one array per dimension. For an image this gives the row (y) and column (x) indices of all the non-zero pixels. https://numpy.org/doc/stable/reference/generated/numpy.nonzero.html<br>
`np.ceil(a)`, `np.log(a)`: element-wise ceiling and natural logarithm.<br>

In [ ]:
# quickstart COLAB
from pathlib import Path
if not Path('lighthouse.png').exists():
  !git clone --quiet https://github.com/tavisualcomputing/viscomp2026/
  %cd /content/viscomp2026/Exercises/W3

## EXERCISE 1 - SETUP CODE

The following cell installs required packages into the virtual machine that will run your code

In [ ]:
%pip install scipy scikit-image matplotlib ipywidgets

Import packages to read videos, make interactive widgets and plot images.

In [ ]:
from skimage.io import imread
from scipy.ndimage import convolve, shift, label
from matplotlib.pyplot import imshow
from ipywidgets import interactive
from IPython.display import display
import matplotlib as mpl
import numpy as np

mpl.rc('image', cmap='gray')  # tell matplotlib to use gray shades for grayscale images

Reading the test image lighthouse.png in `test_im`.

In [ ]:
test_im = np.array(imread("lighthouse.png", as_gray=True), dtype=float)  # This time the image is floating point 0.0 to 1.0!
height, width = test_im.shape
print("Test image shape: ", test_im.shape)
imshow(test_im)

Before starting, here is a function that creates a gaussian filter for you: you might find it handy later.

In [ ]:
def gaussian_filter(size, sigma):
    x, y = np.mgrid[-size//2 + 1:size//2 + 1, -size//2 + 1:size//2 + 1]
    g = np.exp(-((x**2 + y**2)/(2.0*sigma**2)))
    return g/g.sum()

slider = interactive(lambda size, sigma: imshow(gaussian_filter(size, sigma)), {'manual': True, 'manual_name': 'Update image'}, size=(1, 49, 2), sigma=(0.1, 20, 0.1))
slider.children[0].value = 9
slider.children[1].value = 2
display(slider)

## EXERCISE 1 - PUT SOLUTION HERE

## EXERCISE 2 - PUT SOLUTION HERE

In [ ]:
def perform_all_canny(hyperparam1, hyperparam2, <all your hyperparameters here>):
    """
    You can copy inside here all the code you produce in Exercise 2, and take hyperparameters as input to the function.
    Please remove all the imshow function calls in the middle.
    In the end, return the output of the canny edge detection.
    """
    
    ...
    
    return canny_img

# now fill in this template, you can take the gaussian filter generation slider as example on how to fill it if it is unclear, or you can ask.
slider = interactive(lambda hyperparam1, hyperparam2, <all your hyperparameters here>: imshow(perform_all_canny(hyperparam1, hyperparam2, <all your hyperparameters here>)), {'manual': True, 'manual_name': 'Update image'}, hyperparam1=(hyperparam1_min, hyperparam1_max, hyperparam1_step), hyperparam2=(hyperparam2_min, hyperparam2_max, hyperparam2_step), <all your hyperparameters here>)
slider.children[0].value = hyperparam1_initialvalue
slider.children[1].value = hyperparam2_initialvalue
<all your hyperparameters here>

display(slider)

## EXERCISE 3 - SETUP CODE

Synthetic test data: 100 points close to the line $y = 0.6x + 2$ (inliers) plus 60 outliers scattered in the upper-left part of the plot, and a helper to draw a line given in normal form.

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
n_inliers, n_outliers = 100, 60
x_in = rng.uniform(0, 10, n_inliers)
y_in = 0.6 * x_in + 2 + rng.normal(0, 0.15, n_inliers)
outliers = rng.uniform([0, 5], [6, 10], size=(n_outliers, 2))  # outliers concentrated in the upper left
points = np.concatenate([np.stack([x_in, y_in], axis=1), outliers])  # shape (N, 2), each row is (x, y)
gt_inlier_mask = np.arange(len(points)) < n_inliers


def draw_line(ax, n, d, points, **kwargs):
    # draws the segment of the line n^T p + d = 0 that spans the projection of `points` onto it
    direction = np.array([-n[1], n[0]])
    foot = -d * n  # point on the line closest to the origin
    t = (points - foot) @ direction
    a, b = foot + t.min() * direction, foot + t.max() * direction
    ax.plot([a[0], b[0]], [a[1], b[1]], **kwargs)


fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*points[gt_inlier_mask].T, s=10, label='inliers')
ax.scatter(*points[~gt_inlier_mask].T, s=10, label='outliers')
ax.set_aspect('equal')
ax.legend()
plt.show()

## EXERCISE 3 - PUT SOLUTION HERE

In [ ]:
def fit_line_two_points(p1, p2):
    # line through p1 and p2 in normal form: returns unit normal n and offset d with n^T p + d = 0
    # TODO: compute the direction of the line and rotate it by 90 degrees to get the normal
    n = ...
    # TODO: normalize n and compute d
    d = ...
    return n, d


def point_line_distance(points, n, d):
    # points: (N, 2), returns (N,) distances
    # TODO: distance of every point to the line, without a python loop
    return ...


def fit_line_least_squares(points):
    # total least squares: the line passes through the centroid, its normal is the direction of least variance
    centroid = ...  # TODO
    # TODO: use np.linalg.svd on the centered points to find the normal
    n = ...
    d = ...
    return n, d


def ransac_line(points, n_iters, threshold, rng):
    best_inliers = np.zeros(len(points), dtype=bool)
    for _ in range(n_iters):
        # TODO: sample 2 distinct points (skip the iteration if they coincide)
        ...
        # TODO: fit a candidate line and compute its inlier mask (distance < threshold)
        inliers = ...
        # TODO: keep the candidate with the most inliers
        ...
    # TODO: refine the line with least squares on the best inlier set
    n, d = ...
    return n, d, best_inliers

In [ ]:
def ransac_num_iterations(p, outlier_ratio, sample_size):
    # number of samples needed to draw at least one outlier-free sample with probability p
    # TODO
    return ...


outlier_ratio = n_outliers / len(points)
n_iters = ransac_num_iterations(0.99, outlier_ratio, 2)
print(f"outlier ratio {outlier_ratio:.2f} -> {n_iters} iterations")
for e in [0.1, 0.3, 0.5, 0.7, 0.9]:
    print(f"e = {e:.1f}: {ransac_num_iterations(0.99, e, 2):5d} iterations for a line (s=2), "
          f"{ransac_num_iterations(0.99, e, 8):8d} for s=8")

In [ ]:
n_ls, d_ls = fit_line_least_squares(points)
n_rs, d_rs, inliers = ransac_line(points, n_iters, threshold=0.3, rng=np.random.default_rng(1))

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(*points[inliers].T, s=10, c='tab:green', label='RANSAC inliers')
ax.scatter(*points[~inliers].T, s=10, c='tab:gray', label='RANSAC outliers')
draw_line(ax, n_ls, d_ls, points, c='tab:red', label='least squares (all points)')
draw_line(ax, n_rs, d_rs, points, c='tab:blue', label='RANSAC')
ax.set_aspect('equal')
ax.legend()
plt.show()
print(f"RANSAC found {inliers.sum()} inliers, {np.sum(inliers & gt_inlier_mask)} of them are true inliers")

Now apply RANSAC to real edges: we take the edge pixels of lighthouse.png (with the Canny detector of scikit-image, so that this part does not depend on your Exercise 2) and detect several lines one after the other. After each detection, the inliers are removed from the point set before running RANSAC again.

In [ ]:
from skimage.feature import canny

edges = canny(test_im, sigma=3.0)
ys, xs = np.nonzero(edges)
edge_points = np.stack([xs, ys], axis=1).astype(float)  # (x, y) = (column, row)
print("Number of edge pixels:", len(edge_points))

rng = np.random.default_rng(0)
remaining = edge_points
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(edges)
axes[1].imshow(test_im)
for k in range(5):
    n, d, inliers = ransac_line(remaining, n_iters=2000, threshold=1.0, rng=rng)
    draw_line(axes[1], n, d, remaining[inliers], lw=2, label=f"line {k}: {inliers.sum()} inliers")
    remaining = remaining[~inliers]
axes[1].legend(fontsize=8)
plt.show()

Questions to think about:
1. Why is the least-squares line so far off on the synthetic data, while RANSAC is not?
2. How do the threshold $\tau$ and the number of iterations influence the result? What happens if $\tau$ is much too small or much too large?
3. Some of the lines detected in lighthouse.png do not correspond to any straight edge in the image. Why does RANSAC accept them?

## SAMPLE EXAM QUESTION

![alt text](Q1.png "Question 1")

![alt text](Q2.png "Question 2")

![alt text](F2.png "Figure 4")

## Contact

We've included our contact details here in case you are solving this from home and you want to ask a question! Feel free to email us.

Name: Xiaozhong Lyu<br>
E-mail: xiaozhong.lyu@inf.ethz.ch<br>

If your question is very much code-related, we suggest you to make your notebook public and link it to us in your question, so that we can work it out better!